# PANOSETI Cloud Detection — Quickstart

End-to-end tour of the two data paths introduced in PR #8:

| Path | Steps | When to use |
|------|-------|-------------|
| **Fast (in-memory)** | PFF → `slice_to_l1` → `slice_to_l2_cloud` | R&D, parameter sweeps, any time you don't need a stored L1 |
| **Full (disk)** | PFF → `pa-convert` → L0 Zarr → `pa-calibrate` → L1 Zarr → inference | Production, archival, reproducible L1 |

The fast path skips both L0 and L1 materialisation entirely — you get calibrated cloud
scores from a raw PFF sequence in seconds, not minutes.

---
**Requirements:** a `.pffd` observation directory. Any run from BeeGFS works:
```bash
ls /mnt/beegfs/data/L0/
```

In [ ]:
%load_ext autoreload
%autoreload 2

import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from panoseti_analysis.io.bench import stage_timer, summarize
from panoseti_analysis.io.pff import open_pff_product
from panoseti_analysis.paths import REPO_ROOT

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

## §0 Configuration

Set `OBS_DIR` to any `.pffd` run. Discover available products with `PanosetiRun`.

In [ ]:
# ── Point to a real .pffd run ────────────────────────────────────────────────
OBS_DIR = Path("/mnt/beegfs/data/L0/obs_Lick.start_2024-04-19T05:39:33Z.runtype_eng-test.pffd")

# Discover available products
from pypff import PanosetiRun

run = PanosetiRun(str(OBS_DIR))
products = run.list_products()
print("Available products:")
for p in products:
    print(f"  {p}")

# Pick the first img16 product
DP = next((p for p in products if "img16" in p), products[0])
MODULE = int(DP.split("module_")[-1]) if "module_" in DP else 1
print(f"\nUsing: {DP}  (module {MODULE})")

## §1 Inspect the PFF sequence

`open_pff_product` returns a zero-copy mmap-backed `PFFSequence`. No data is loaded yet.

In [ ]:
seq = open_pff_product(OBS_DIR, DP, MODULE)
ts = seq.timestamps()  # int64 ns array, cached
T = len(seq)

print(f"Frames        : {T:,}")
print(f"Duration      : {(ts[-1] - ts[0]) / 1e9:.1f} s  ({(ts[-1] - ts[0]) / 3600e9:.2f} h)")
print(f"Cadence       : {(ts[1] - ts[0]) / 1e3:.0f} µs")
print(f"Frame config  : {seq.frame_config}")
print(f"Files         : {len(seq.file_paths)}")

## §2 Fast path: PFF → in-memory L1 → cloud scores

`slice_to_l1` chains:
1. `sequence_to_dataset(seq, start, stop, step=decimate)` — in-memory L0-layout Dataset
2. `repair_timestamps` → `calibrate_img` — produces `median_subtracted` in RAM

`slice_to_l2_cloud` extends this to cloud inference without any disk write.

### Decimation
With `decimate=D`, every D-th frame is read. For R&D this is usually fine — the cloud
detector operates on 60 s windows; 100 µs × D must stay ≪ 60 s. `decimate=100` →
10 ms cadence, well within that budget.

In [ ]:
from panoseti_analysis.adapters.slice_driver import slice_to_l1

RECIPE = REPO_ROOT / "recipes/img_calib_default.yml"
N_FRAMES = 50_000  # ~5 s at 100 µs cadence
DECIMATE = 10  # read every 10th frame → 1 ms effective cadence

results = []

with stage_timer("slice_to_l1") as r:
    ds_l1 = slice_to_l1(
        seq,
        frame_range=(0, N_FRAMES),
        decimate=DECIMATE,
        recipe=RECIPE,
        fail_on_suspect=False,
    )
results.append(r)

T_l1 = ds_l1.sizes["time"]
mem_mb = ds_l1["median_subtracted"].nbytes / 1e6
print(f"Input frames : {N_FRAMES:,}  (decimate={DECIMATE} → {T_l1:,} output frames)")
print(f"L1 variables : {list(ds_l1.data_vars)}")
print(f"RAM usage    : {mem_mb:.1f} MB")
print(f"Wall time    : {r.seconds:.2f} s")

### Visualise the in-memory L1

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

med = ds_l1["median_subtracted"].values
t_s = (ds_l1["unix_t_ns"].values - ds_l1["unix_t_ns"].values[0]) / 1e9

# Mid-sequence frame
mid = T_l1 // 2
im0 = axes[0].imshow(med[mid], cmap="RdBu_r", vmin=-5, vmax=5, interpolation="nearest")
axes[0].set_title(f"median_subtracted (frame {mid})")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046, label="ADC")

# Time-averaged
axes[1].imshow(np.mean(med, axis=0), cmap="viridis", interpolation="nearest")
axes[1].set_title("Time average")
axes[1].axis("off")

# Pixel RMS time series
rms = np.mean(np.abs(med), axis=(1, 2))
axes[2].plot(t_s, rms, lw=0.8, color="steelblue")
axes[2].set_xlabel("t (s)")
axes[2].set_ylabel("|median_subtracted| mean")
axes[2].set_title("Spatial mean signal vs time")

plt.suptitle(f"In-memory L1  ({T_l1} frames, decimate={DECIMATE})", y=1.01)
plt.tight_layout()
plt.show()

### Cloud detection on the in-memory L1

In [ ]:
from panoseti_analysis.adapters.slice_driver import slice_to_l2_cloud
from panoseti_analysis.algorithms.cloud_detector import CloudDetectionV2
from panoseti_analysis.config.models import CloudInferParams
from panoseti_analysis.io.models import load_classifier

MODEL_PT = REPO_ROOT / "ml/cloud-detection/models/cloud_detector_v2_legacy.pt"
INFER_PARAMS = CloudInferParams(cadence_s=10.0, window_s=60.0, n_stack=10)

if not MODEL_PT.exists():
    print(f"Model not found: {MODEL_PT}")
    print("Train one with notebook 02 or 03, or point MODEL_PT to an existing .pt file.")
else:
    model, bundle = load_classifier(MODEL_PT, model=CloudDetectionV2())
    model.eval()
    print(f"Model: {bundle.model_name} v{bundle.model_version}")

    with stage_timer("slice_to_l2_cloud") as r:
        ds_l2 = slice_to_l2_cloud(
            seq,
            model,
            frame_range=(0, N_FRAMES),
            decimate=DECIMATE,
            recipe=RECIPE,
            infer_params=INFER_PARAMS,
        )
    results.append(r)

    scores = ds_l2["cloud_score"].values
    labels = ds_l2["cloud_label"].values
    t_l2 = (ds_l2["unix_t_ns"].values - ds_l2["unix_t_ns"].values[0]) / 1e9

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(t_l2, scores, lw=1.0, color="steelblue", label="cloud_score")
    ax.axhline(
        INFER_PARAMS.threshold,
        color="tomato",
        ls="--",
        lw=0.8,
        label=f"threshold={INFER_PARAMS.threshold}",
    )
    ax.fill_between(t_l2, labels.astype(float), alpha=0.2, color="tomato", label="cloudy")
    ax.set_xlabel("t (s)")
    ax.set_ylabel("P(cloudy)")
    ax.set_title("Cloud score — in-memory fast path")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print(f"Windows: {len(scores)}  cloudy: {labels.sum()} ({labels.mean():.1%})")
    print(f"Elapsed: {r.seconds:.2f} s")

## §3 Comparison: fast path vs full materialization

The full path writes L0 Zarr (convert), then L1 Zarr (calibrate), then runs inference on
the materialized store. The fast path does all three steps in memory.

This cell runs both and compares wall times on the same `N_FRAMES` window.

In [ ]:
import tempfile

from panoseti_analysis.adapters.calibrate import run_calibrate
from panoseti_analysis.adapters.convert import run_convert

bench = []  # BenchResult list for both paths

# ── Fast path (already timed above) ──────────────────────────────────────────
with stage_timer("fast: slice_to_l1 (in-memory)") as r_fast:
    _ = slice_to_l1(
        seq, frame_range=(0, N_FRAMES), decimate=DECIMATE, recipe=RECIPE, fail_on_suspect=False
    )
r_fast.bytes_out = ds_l1["median_subtracted"].nbytes
bench.append(r_fast)

# ── Full path: convert → calibrate on disk ───────────────────────────────────
with tempfile.TemporaryDirectory(prefix="pano_bench_") as tmpdir:
    l0_dir = Path(tmpdir) / "L0"
    l1_dir = Path(tmpdir) / "L1"

    with stage_timer("full: pa-convert (PFF → L0 Zarr)") as r_convert:
        records = run_convert(OBS_DIR, l0_dir, checksum=False)
    bench.append(r_convert)

    l0_stores = list(l0_dir.glob("*.zarr"))
    if l0_stores:
        with stage_timer("full: pa-calibrate (L0 → L1 Zarr)") as r_calib:
            run_calibrate(l0_stores[0], l1_dir, recipe=RECIPE, fail_on_suspect=False)
        bench.append(r_calib)

print(summarize(bench))

total_full = sum(r.seconds for r in bench[1:])
speedup = total_full / bench[0].seconds
print(f"\nSpeedup: fast path is {speedup:.1f}× faster than full materialisation")
print("(for this {N_FRAMES}-frame window — speedup grows with decimate factor)")

## §4 Decimation sweep

How much faster does `slice_to_l1` get as we decimate more aggressively?
The compute (calibration) drops linearly with `decimate`; the mmap read also drops
because we skip most frame reads.

In [ ]:
import pandas as pd

sweep_results = []
for dec in [1, 5, 10, 50, 100]:
    with stage_timer(f"decimate={dec}") as r:
        ds = slice_to_l1(
            seq, frame_range=(0, N_FRAMES), decimate=dec, recipe=RECIPE, fail_on_suspect=False
        )
    r.bytes_out = ds["median_subtracted"].nbytes
    sweep_results.append(
        {
            "decimate": dec,
            "frames_out": ds.sizes["time"],
            "time_s": r.seconds,
            "output_MB": r.bytes_out / 1e6,
        }
    )
    print(
        f"decimate={dec:3d}: {ds.sizes['time']:5d} frames, {r.seconds:.3f} s, {r.bytes_out / 1e6:.1f} MB"
    )

df_sweep = pd.DataFrame(sweep_results)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(df_sweep["decimate"], df_sweep["time_s"], "o-", color="steelblue")
ax1.set_xlabel("decimate factor")
ax1.set_ylabel("wall time (s)")
ax1.set_title("slice_to_l1 wall time vs decimate")
ax1.set_xscale("log")

ax2.plot(df_sweep["decimate"], df_sweep["output_MB"], "o-", color="tomato")
ax2.set_xlabel("decimate factor")
ax2.set_ylabel("output size (MB)")
ax2.set_title("L1 RAM usage vs decimate")
ax2.set_xscale("log")

plt.tight_layout()
plt.show()

## §5 New features cheatsheet

### Fast in-memory path
```python
from panoseti_analysis.io.pff import open_pff_product
from panoseti_analysis.adapters.slice_driver import slice_to_l1, slice_to_l2_cloud

seq = open_pff_product(obs_dir, dp, module)

# By frame index:
ds_l1 = slice_to_l1(seq, frame_range=(0, 10_000), decimate=10, recipe=recipe)

# By timestamp (nanoseconds):
ds_l1 = slice_to_l1(seq, time_range=(start_ns, stop_ns), decimate=10, recipe=recipe)

# All the way to cloud scores:
ds_l2 = slice_to_l2_cloud(seq, model, frame_range=(0, 10_000), decimate=10,
                           recipe=recipe, infer_params=CloudInferParams(cadence_s=10.0))
```

### read_stride for decimated disk L1
```python
# Produce a decimated L1 store on disk (faster write, smaller file):
pa-calibrate l0_store l1_store --read-stride 10
# Or via Python:
from panoseti_analysis.adapters.calibrate import run_calibrate
run_calibrate(l0_store, l1_store, read_stride=10)
```

### Skip L0 materialisation entirely
```python
from panoseti_analysis.adapters.recipe_driver import run_pipeline
outputs = run_pipeline(obs_dir, out_dir, recipe=recipe, skip_l0_materialize=True)
# L1 is written; L0 is never written.
```

### Benchmarking
```python
from panoseti_analysis.io.bench import stage_timer, summarize

with stage_timer("my_step") as r:
    do_work()
r.bytes_out = result_nbytes
print(summarize([r]))
```

### NCCL/RDMA for 2-node DDP
```yaml
# recipes/ml/vae_train_v1.yml — opt in explicitly:
scaling:
  num_workers: 2
  accelerator_type: "A6000"
  allow_multinode: true   # required when num_workers > 1
```
```bash
bash cluster/ral_up.sh   # injects NCCL_IB_HCA=mlx5_0 etc on all nodes
pa-train-vae --launcher attach --recipe recipes/ml/vae_train_v1.yml ...
```